# 5 · The shape of a population

**The problem.** Test data, sample data, synthetic data: each needs not just valid contacts but a believable *mix* of
them. Three in ten are seniors; most have a phone. A validator can't express "most": no one contact breaks it. The mix
is a property of the **population**.

A **pattern** is a predicate whose rule also weighs alternatives. `Distributions.Choices` holds weighted **arms**; the
pattern is still a predicate, so it still checks and selects, and now it also weighs.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Distributions, Generators, Predicates, Validators } from "@mbse/patterns";
import { Stores } from "@mbse/schemas/Framework";
import { Contact, Directory, book, listed } from "./toolkit.ts";

const person = E.variable("person");
const HasName = new Predicates.OfPredicate.Builder().name("HasName").symbols({ person: Contact })
  .parameters((x) => x.name("name")).requires(person.name.eq(E.variable("name"))).create();

## Step 1: weighted alternatives

Each arm has a weight and conditions. The choices hold when enough arms hold: at least one, by default.

In [2]:
const Names = new Predicates.OfPredicate.Builder().name("Names").symbols({ person: Contact }).requires(
  new Distributions.Choices.Builder().arms(
    (a) => a.weight(10).requires(HasName.call(person, "Ann")),
    (a) => a.weight(5).requires(HasName.call(person, "Bob")),
    (a) => a.weight(15).requires(HasName.call(person, "Cy")),
  ).create()).create();
const store = book();
listed(store, ...["Ann", "Bob", "Cy", "Zed"].map((n) => store.Contact().name(n).create()));
console.log(Validators.Validate(store, [Names]).Reachable(Directory, store.singleton("book.Directory")));

[ "person=Contact#4: 'Names' does not hold" ]


As a rule, the pattern says "the name is one of these", and Zed breaks it.

## Step 2: what a match weighs

`Evaluator.weigh` gives what a match weighs under the rule: nothing unless it holds, then its arm's weight.

In [3]:
const evaluate = new Predicates.Evaluator(store);
console.log(Object.fromEntries([...store.extent("Contact")].map((c: any) => [c.name, evaluate.weigh(Names.rule, { person: c })])));

{ Ann: 10, Bob: 5, Cy: 15, Zed: 0 }


## Step 3: counts and precedence

`.count(...)` says how many arms must hold, as a condition on the number. `.decreasing()` puts the arms in order: a
match falls under the first that holds, so put the more specific arms first.

In [4]:
const overlap = () => new Distributions.Choices.Builder().arms(
  (a) => a.weight(3).requires(person.has("name")),
  (a) => a.weight(2).requires(HasName.call(person, "Ann")));
const ann = [...store.extent("Contact")].find((c: any) => c.name === "Ann");
console.log(
  evaluate.weigh(overlap().create(), { person: ann }), // every arm that holds: 3 + 2
  evaluate.weigh(overlap().decreasing().create(), { person: ann }), // the first that holds
  evaluate.run(overlap().count((c) => c.eq(1n)).create(), { person: ann })); // exactly one arm: not Ann, who holds both

5 3 false


## Step 4: sampling the store

`Generators.Sample` draws the store's own matches, each as often as it weighs, from a random source you give it. The
same seed gives the same draws, in TypeScript and Python alike:

In [5]:
const drawn = Generators.Sample(store, Names, new Stores.PCG32(42n));
const counts = new Map<string, number>();
for (let i = 0; i < 3000; i++) {
  const name = (drawn.next().value as any).person.name;
  counts.set(name, (counts.get(name) ?? 0) + 1);
}
console.log(counts);

Map(3) { "Ann" => 1004, "Cy" => 1513, "Bob" => 483 }


About 10 : 5 : 15, and never Zed.

## Why it works this way

- **A pattern is a predicate.** It validates, queries and weighs with one rule, so the data you generate from it and
  the data you check against it can't disagree.
- **Weights are relative.** Arms needn't sum to one: 10, 5 and 15 are the same mix as 1/3, 1/6 and 1/2.
- **The random source is the caller's.** A store is data access alone; seeds belong to whoever draws.